# 01 - Preprocessing & EDA

FinQA dev split: what the raw data looks like, what's wrong with it, and how it
becomes a retrievable corpus.

All logic lives in `src/data/*` and `src/chunking/row_level.py`; this notebook
only calls into it and plots the results. To regenerate the processed files
non-interactively: `python scripts/run_preprocessing.py`.


In [ ]:
# notebooks are thin: every function called here is imported from src/
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd

from src.data.loader import flatten_examples, load_raw_finqa

data = load_raw_finqa()
df = pd.DataFrame(flatten_examples(data))
print(f"{len(data)} QA examples, {df['filename'].nunique()} unique source pages")
df.head()


## 1. The shape of the raw data

One row per *question*, not per document. The page content (`pre_text`, `table`,
`post_text`) repeats once per question asked about that page, which is why 883
examples collapse to 299 documents later.


In [ ]:
df.info()


In [ ]:
import matplotlib.pyplot as plt
from collections import Counter

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

axes[0, 0].hist(df["n_pre_text_lines"], bins=30, alpha=0.6, label="pre_text")
axes[0, 0].hist(df["n_post_text_lines"], bins=30, alpha=0.6, label="post_text")
axes[0, 0].set_title("Text lines per document")
axes[0, 0].legend()

axes[0, 1].hist(df["n_table_rows"], bins=20, color="teal")
axes[0, 1].set_title("Table rows per document")

axes[0, 2].hist(df["n_table_cols"], bins=15, color="orange")
axes[0, 2].set_title("Table columns per document")

axes[1, 0].hist(df["question_word_len"], bins=25, color="purple")
axes[1, 0].set_title("Question length (words)")

op_counts = Counter(op for ops in df["ops"] for op in ops)
axes[1, 1].bar(op_counts.keys(), op_counts.values(), color="crimson")
axes[1, 1].set_title("Arithmetic operation frequency")
axes[1, 1].tick_params(axis="x", rotation=45)

axes[1, 2].hist(df["n_gold_table_rows"], bins=10, alpha=0.6, label="table rows")
axes[1, 2].hist(df["n_gold_text_rows"], bins=10, alpha=0.6, label="text rows")
axes[1, 2].set_title("Gold rows needed per question")
axes[1, 2].legend()

plt.tight_layout()
plt.show()


## 2. The noise, and why lines can't just be dropped

FinQA's text is pre-tokenized, so punctuation floats away from its word (`inc .`,
`$ 2457`, `( 1 )`) and a few hundred lines are nothing but a lone period.

The tempting fix is to filter those junk lines out. That's the bug this project
already hit: `ann_text_rows` holds *positional* indices into `pre_text + post_text`,
so deleting line 4 silently repoints every gold label after it. `index_preserving_clean`
therefore cleans text in place and only *flags* `is_noise`, keeping length and order.


In [ ]:
import re

from src.data.cleaning import clean_line, index_preserving_clean, is_noise_line

lone_punct = sum(1 for ex in data for l in ex["pre_text"] + ex["post_text"] if is_noise_line(l))
spaced_dollar = sum(1 for ex in data for l in ex["pre_text"] + ex["post_text"] if re.search(r"\$ \d", l))
spaced_paren = sum(1 for ex in data for l in ex["pre_text"] + ex["post_text"]
                   if re.search(r"\( \d", l) or re.search(r"\d \)", l))

plt.figure(figsize=(6, 4))
plt.bar(["Lone punctuation\nlines", "Spaced '$ 123'\npatterns", "Spaced '( 1 )'\npatterns"],
        [lone_punct, spaced_dollar, spaced_paren], color="darkred", alpha=0.7)
plt.title(f"Noise patterns across {len(data)} documents")
plt.ylabel("occurrence count")
plt.tight_layout()
plt.show()

raw_line = data[0]["pre_text"][3]
print("before:", raw_line)
print("after: ", clean_line(raw_line))


In [ ]:
lines = data[0]["pre_text"]
cleaned = index_preserving_clean(lines)

print(f"in {len(lines)} lines -> out {len(cleaned)} lines (must be equal)")
print(f"{sum(c['is_noise'] for c in cleaned)} flagged as noise, 0 removed")
cleaned[3]


## 3. Page duplication

Same `filename` means byte-identical page content, so the corpus is 299 pages,
not 883. Each document keeps a `qa_ids` backlink to the questions asked about it.


In [ ]:
from src.data.reconstruction import build_documents, group_by_filename

by_file = group_by_filename(data)
sample_fname = next(f for f, exs in by_file.items() if len(exs) > 1)
exs = by_file[sample_fname]

print(f"{sample_fname} -> {len(exs)} questions")
print("pre_text identical across them?", all(e["pre_text"] == exs[0]["pre_text"] for e in exs))
print("table identical across them?  ", all(e["table"] == exs[0]["table"] for e in exs))
for e in exs:
    print("  -", e["qa"]["question"])


In [ ]:
documents = build_documents(data)
print(f"{len(documents)} documents, {sum(len(d['qa_ids']) for d in documents)} questions retained")

docs_df = pd.DataFrame([{
    "doc_id": d["doc_id"],
    "n_pre_text": len(d["pre_text"]),
    "n_post_text": len(d["post_text"]),
    "n_noise_lines": sum(1 for x in d["pre_text"] + d["post_text"] if x["is_noise"]),
    "table_shape": f"{len(d['table'])}x{len(d['table'][0]) if d['table'] else 0}",
    "n_questions": len(d["qa_ids"]),
} for d in documents])
docs_df.head(10)


## 4. Row-level chunks and the gold mapping

Table rows are linearized into sentences (`'Visa Inc.(1): Payments Volume is
$2,457; Cards is 1,592.'`) with `row_index` counting from 1, because row 0 is the
header and that's how `ann_table_rows` indexes. Text lines keep their position in
`pre_text + post_text`. Because both indices survive cleaning, a gold reference
is just a formatted `chunk_id`, and the integrity check below should read 0.


In [ ]:
from src.chunking import build_all
from src.chunking.row_level import build_row_level_chunks
from src.eval.gold_mapping import build_eval_dataset

chunks = build_all(documents, build_row_level_chunks)
print(f"{len(chunks)} chunks, {sum(c['is_noise'] for c in chunks)} flagged noise")

eval_examples, missing = build_eval_dataset(data, documents, chunks)
print(f"eval_dataset: {len(eval_examples)} examples, {missing} unresolved gold references")
assert missing == 0


In [ ]:
# trace one question end to end: raw annotation -> chunk text
ex = data[0]
print("question:      ", ex["qa"]["question"])
print("ann_table_rows:", ex["qa"].get("ann_table_rows"))
print("ann_text_rows: ", ex["qa"].get("ann_text_rows"))

by_id = {c["chunk_id"]: c for c in chunks}
for gid in eval_examples[0]["gold_chunk_ids"]:
    print(f"\n{gid}\n  -> {by_id[gid]['text'][:200]}")


## 5. Persist

`scripts/run_preprocessing.py` does exactly this and nothing more, so the files
can be rebuilt without opening a notebook.


In [ ]:
from src.utils.io import save_jsonl

save_jsonl(documents, "data/processed/documents.jsonl")
save_jsonl(chunks, "data/processed/chunks.jsonl")
save_jsonl(eval_examples, "data/processed/eval_dataset.jsonl")
print("wrote documents.jsonl, chunks.jsonl, eval_dataset.jsonl")
